<div>
    <img style="float:right; width:210px" src="images/snext-logo.png"/>
    <div style="float:left;"><h1>Introduction to Python for Data Science</h1></div>
</div>

---
# Notebook 3: Pandas
This notebook introduces the `pandas` package as a convenient toolset to work with tabular data.  
We build upon the basics of Python and pandas to explore data from NASA's **Near‑Earth Object Web Service (NeoWs)**. We will learn how to retrieve data from a web **API**, convert the raw JSON into a tidy pandas DataFrame, filter and analyze the data, compute statistics, and create simple visualizations.

## Contents

[1. Importing data from APIs](#chapter1)  
[2. Introduction to DataFrames](#chapter2)  
[3. Simple data visualization](#chapter3)  
---

# 1. Importing data from APIs <a id="chapter1"/>

Most modern datasets live on the web behind **Application Programming Interfaces (APIs)**. An API is a contract, a set of rules, that allow different software applications to communicate with each other.

<figure style="text-align:center">
  <img
    src="https://media2.dev.to/dynamic/image/width=1000,height=420,fit=cover,gravity=auto,format=auto/https%3A%2F%2Fdev-to-uploads.s3.amazonaws.com%2Fuploads%2Farticles%2F15k1xkvzxry2jif8vems.png"
    alt="Near-Earth Objects illustration"
    style="max-width: 100%; height: auto;"
  />
  <figcaption>
    <em>Figure.</em> API (illustrative image). Source: DEV Community image asset,
    accessed 14 Oct 2025.
    <br/>
    Original file:
    <a href="https://dev-to-uploads.s3.amazonaws.com/uploads/articles/15k1xkvzxry2jif8vems.png">
      dev-to-uploads.s3.amazonaws.com/.../15k1xkvzxry2jif8vems.png
    </a>
  </figcaption>
</figure>

### 1.1 NASA's Near‑Earth Object Web Service (NeoWs)

We'll start this session by using [REST-APIs](https://en.wikipedia.org/wiki/Representational_state_transfer) to retrieve some data. In short, when using a REST API, we use the same methods as a browser does, when retrieving a webpage. But instead of an HTML description of a webpage, we retrieve the data. 
For practising , [this](https://github.com/public-apis/public-apis) is a list of publicly available APIs. In this notebook we are going to use `NeoWs`, the **Near‑Earth Object Web Service (NeoWs)** lets you search for asteroids based on their **closest approach date** to Earth

NeoWs offers three endpoints:
- **Feed**: returns asteroids approaching Earth within a date range (`/neo/rest/v1/feed`).
- **Lookup**: returns detailed information for a specific asteroid by ID (`/neo/rest/v1/neo/<id>`).
- **Browse**: returns a paginated listing of all known asteroids (`/neo/rest/v1/neo/browse`).

To access the API you need an API key.  NASA provides a public `DEMO_KEY` which allows up to 30 requests per hour. You can request your own key at [api.nasa.gov](https://api.nasa.gov) for higher rate limits, but the demo key suffices for this exercise.

We will use the **feed** endpoint to retrieve all near‑Earth objects between two dates.  Each date can return multiple asteroids.
First let's import the `requests` package and make a request.  Replace `start_date` and `end_date` with any dates you are interested in.

In [ ]:

import requests  # package to send http queries to the API

# choose a 7‑day interval.  NASA's feed endpoint accepts a maximum of 7 days per request.
# feel free to change these dates to explore other periods.
start_date = '2025-10-7'
end_date   = '2025-10-14'
api_key    = 'DEMO_KEY'  # replace with your personal key if you have one, API keys identify who is calling the API and help providers enforce fair usage.

# build the URL
# this is an f-string, anything inside the curly braces {} is replaced by the value of the python variable. It is a clean way to insert variables into text.
# After the '?' we pass *query parameters* that the API expects: start_date=..., end_date=..., api_key=...
url = f'https://api.nasa.gov/neo/rest/v1/feed?start_date={start_date}&end_date={end_date}&api_key={api_key}'

# send a GET request to the API: A GET request is used to retrieve data
res = requests.get(url)

# modern APIs return the data in JSON text format, json() function parses that format and converts it to a Python object
raw_data = res.json()

# uncomment the next line to inspect the structure of the response
raw_data

---
## <span style="color:#FF5D02;">Assignment: Analyze data structure </span>

The retrieved raw data should look like this:

```
{
  "links": {"next": "...", "prev": "...", "self": "..."},
  "element_count": 27,
  "near_earth_objects": {
    "2021-09-13": [
      {
        "id": "2452807",
        "name": "452807 (2006 KV89)",
        "estimated_diameter": {
          "kilometers": {
            "estimated_diameter_min": 0.1319,
            "estimated_diameter_max": 0.2951
          }
        },
        "is_potentially_hazardous_asteroid": true,
        "close_approach_data": [
          {
            "close_approach_date": "2021-09-13",
            "relative_velocity": {"kilometers_per_second": "8.4802313878"},
            "miss_distance": {"kilometers": "68171991.812267909"},
            "orbiting_body": "Earth"
          }
        ]
      },
      ...
    ],
    "2021-09-14": [ ... ],
    ...
  }
}
    
```

Please take a minute to describe for yourself what data structures you recognize! Hint: they're nested inside each other.


**Hints**

Is it a list of dictionaries? A dictionary with keys that contain lists as values? A dictionary of dictionaries?

Recap how we accessed data inside dictionaries and lists.  
Try to access some of the data fields with the syntax you've learned previously to access dictionaries and lists:

In [ ]:
raw_data[...][...]...

**Tip for beginners**
In Jupyter, use the built-in interactive JSON viewer to explore API data responses without getting overwhelmed!

In [ ]:
from IPython.display import JSON

display(JSON(raw_data))

Expand the following two cells to see the solution!

In [ ]:
raw_data["near_earth_objects"]["2025-10-14"][0] # this returns the first asteroid in the list on the 14.October

In [ ]:
raw_data["near_earth_objects"]["2025-10-14"][0]["is_potentially_hazardous_asteroid"] # this returns whether that astroid is flagged as hazardous

---

# 2. Introduction to DataFrames<a id="chapter2"/>
You probably noticed that working purely with lists of dictionaries and such is not 
Pandas are the central tool for reading and manipulating data in Python. For our purposes, the `DataFrames` data structure is the most important:

> **DataFrame** is a 2-dimensional labeled data structure with columns of potentially different types.  
> You can think of it like a spreadsheet or SQL table [...]. It is generally the most commonly used pandas object.  
> [(Source)](https://pandas.pydata.org/pandas-docs/stable/user_guide/dsintro.html)

Dataframes contain rows and columns and distinguish between regular data and indices - columns that contain an unique identifier for each row.
I.e. our just imported covid dataset would look like this:

<img src="images/dataframe.png"/>  

The full documentation can be found [here](https://pandas.pydata.org/pandas-docs/stable/reference/frame.html).

Let's explore step by step, how data frames make life easier when exploring data. The function in the next cell is meant to simplify our data making our exploration easier , go ahead and run it.

In [ ]:
# Helper to Flatten dict-of-dates → list of asteroid dicts with a 'date' field
def simplify_data(nasa_raw):
    neo_by_date = nasa_raw.get("near_earth_objects", {})
    rows = []

    for date_key, objects in neo_by_date.items():
        for obj in objects:
            base = {
                "id": obj.get("id"),
                "neo_reference_id": obj.get("neo_reference_id"),
                "name": obj.get("name"),
                "nasa_jpl_url": obj.get("nasa_jpl_url"),
                "absolute_magnitude_h": obj.get("absolute_magnitude_h"),
                "is_potentially_hazardous_asteroid": obj.get("is_potentially_hazardous_asteroid"),
                "is_sentry_object": obj.get("is_sentry_object"),
                "sentry_data": obj.get("sentry_data"),
                "date": date_key,
                "links.self": (obj.get("links") or {}).get("self"),
                "estimated_diameter": obj.get("estimated_diameter"),
            }

            cad = (obj.get("close_approach_data") or [{}])[0]  # first entry or empty
            row = base | {
                "close_approach_date": cad.get("close_approach_date"),
                "close_approach_date_full": cad.get("close_approach_date_full"),
                "epoch_date_close_approach": cad.get("epoch_date_close_approach"),
                "orbiting_body": cad.get("orbiting_body"),
                "relative_velocity": cad.get("relative_velocity", {}),
                "miss_distance": cad.get("miss_distance", {}),
            }
            rows.append(row)

    return {"near_earth_objects": rows}



#Convert all columns in the given DataFrame that contain numeric strings into actual numeric (float) types, wherever possible.
def cast_dtypes(df):
    """
    Cast NASA NEO dataset columns to their correct pandas dtypes.
    Use this right after you create your DataFrame.
    """
    dtype_map = {
        # text / identifiers
        "id": "string",
        "neo_reference_id": "string",
        "name": "string",
        "nasa_jpl_url": "string",
        "orbiting_body": "string",
        "sentry_data": "string",
        "links.self": "string",

        # date and time
        "date": "string",  # can later be converted to datetime if needed
        "close_approach_date": "string",
        "close_approach_date_full": "string",
        "epoch_date_close_approach": "Int64",  # large integer timestamp

        # booleans
        "is_potentially_hazardous_asteroid": "boolean",
        "is_sentry_object": "boolean",

        # numeric
        "absolute_magnitude_h": "float64",
        "estimated_diameter.kilometers.estimated_diameter_min": "float64",
        "estimated_diameter.kilometers.estimated_diameter_max": "float64",
        "estimated_diameter.meters.estimated_diameter_min": "float64",
        "estimated_diameter.meters.estimated_diameter_max": "float64",
        "estimated_diameter.miles.estimated_diameter_min": "float64",
        "estimated_diameter.miles.estimated_diameter_max": "float64",
        "estimated_diameter.feet.estimated_diameter_min": "float64",
        "estimated_diameter.feet.estimated_diameter_max": "float64",
        "relative_velocity.kilometers_per_second": "float64",
        "relative_velocity.kilometers_per_hour": "float64",
        "relative_velocity.miles_per_hour": "float64",
        "miss_distance.astronomical": "float64",
        "miss_distance.lunar": "float64",
        "miss_distance.kilometers": "float64",
        "miss_distance.miles": "float64",
    }

    # Safely cast columns if they exist
    for col, dtype in dtype_map.items():
        if col in df.columns:
            try:
                df[col] = df[col].astype(dtype)
            except Exception:
                # fallback for strings pretending to be numbers
                if "float" in dtype or "int" in dtype:
                    df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

In [ ]:
import pandas
import requests

raw_data_full = requests.get(url).json()  # fetch data
#helper function 1
raw_data_simplified= simplify_data(raw_data)
print(raw_data_simplified)
df_preprocessed = pandas.json_normalize(raw_data_simplified["near_earth_objects"])     # raw_data["data"] contains the relevant datatable as a list of dictionaries [{},{},{},{},....], see above
#helper function 2
df = cast_dtypes(df_preprocessed)

In [ ]:
# let's have a look at our brand new dataframe...
df.columns

In [ ]:
df.head(5)   # shows the first n rows

In [ ]:
# let's simplify a little

# drop some columns to make handling easier
df = df.drop([ 'links.self',
       'estimated_diameter.meters.estimated_diameter_min',
       'estimated_diameter.meters.estimated_diameter_max',
       'estimated_diameter.miles.estimated_diameter_min',
       'estimated_diameter.miles.estimated_diameter_max',
       'estimated_diameter.feet.estimated_diameter_min',
       'estimated_diameter.feet.estimated_diameter_max',
      ], axis="columns")

# simplify column names
df = df.rename(columns={
    'nasa_jpl_url': 'jpl_url',
    'is_potentially_hazardous_asteroid': 'is_hazardous',
    'close_approach_date': 'approach_date',

    # Diameter columns
    'estimated_diameter.kilometers.estimated_diameter_min': 'diameter_min_km',
    'estimated_diameter.kilometers.estimated_diameter_max': 'diameter_max_km',

    # Velocity columns
    'relative_velocity.kilometers_per_second': 'velocity_km_s',
    'relative_velocity.kilometers_per_hour': 'velocity_km_h',
    'relative_velocity.miles_per_hour': 'velocity_mph',

    # Miss distance columns
    'miss_distance.astronomical': 'miss_distance_au',
    'miss_distance.lunar': 'miss_distance_lunar',
    'miss_distance.kilometers': 'miss_distance_km',
    'miss_distance.miles': 'miss_distance_miles'
})

In [ ]:
df.head(5) # let's look again

In [ ]:
df.count()   # shows the number of valid entries per column

## 2.1. Preparing the dataset

Before working with the data, we usually want to remove/rename some columns, sort the data, apply filters or partition the data.
In this chapter we'll briefly walk the some commonly used functions to prepare datasets.

> **Important**: All edits to the Dataframe create a copy with the changes, if you don't explicitly force the function to apply the changes directly ("inplace"). If you don't force inplace editing, the original DataFrame remains unchanged. So you usually have two options to apply changes:  
>
> `df = df.change_something(...)             # assign the copy with the change to the original variable`  
> `df.change_something(..., inplace=True)    # apply the change to the original dataframe`

In [ ]:
# Let's prepare the dataframe...

df = df.sort_values("date", ascending=True)     # sort data ascending
df = df.set_index("date")                       # set date column as unique identifier for records (index)

In [ ]:
df.head(50) # check result

## 2.2. Selections and filtering

Dataframes generally accept filters/selections in the format `df[row_filter, column_filter]`. Rows can be selected via a boolean mask or by using the index; columns can be addressed by name. Below are a few ways to work with the NEO dataset.
When you write df[rows, columns] in pandas, the part before the comma selects which rows to keep and the part after the comma specifies which columns to return

### Select by True/False vector

In [ ]:
# A boolean mask indicating which asteroids are classified as potentially hazardous
df.is_hazardous

In [ ]:
# Select astroids with diameter above 0.1km
biggest = df[df.diameter_max_km  > 0.1]  # filter rows
biggest.sort_values("diameter_max_km", ascending=False)                 # show dataframe, sorted by biggest astroid first

In [ ]:
df[(df.diameter_max_km > 0.1) & (df.velocity_km_s > 6.0)]        # Use bool algebra operators & ("and") and | ("or") to combine filters

In [ ]:
df[(df.is_hazardous == "True") | (df.velocity_km_s > 15)]              # Important: don't forget the brackets

### Select by naming relevant rows, columns

In [ ]:
# Select using function .loc[list of row_indexes, list of column names]:
# This line retrieves the value(s) from the column "is_hazardous" for all rows where the index label equals '2025-10-14'.
# In this DataFrame, the index is the "date" column — so this selects all asteroids that had a close approach on October 14, 2025,  and shows whether each of them is marked as hazardous (True/False).
df.loc['2025-10-14',"is_hazardous"]

In [ ]:
#retrieve all the astroids with the data 14.10 and show their name and velocity
df.loc['2025-10-14', ['name', 'velocity_km_s']]
#if you want to select a non index
df.loc[df['is_hazardous'] == False, ['name', 'velocity_km_s']]

In [ ]:
df.loc["2025-10-07":"2025-10-14", : ]  #It retrieves all columns (:) for every row where the index value  falls between '2020-09-01' and '2020-09-07', inclusive.

In [ ]:
#the first list ["2025-10-13", "2025-10-14"] tells pandas which rows to include here, those are the close-approach dates of asteroids, the second list ["name", "velocity_km_s"] tells pandas which columns to include

df.loc[["2025-10-13", "2025-10-14"], ["name", "velocity_km_s"]]

---
## <span style="color:#FF5D02;">Assignment: Data selection</span>

Generate a new DataFrame that only contains only hazardous asteroids and display their name and velocity.

**Hints**

In [ ]:
# Solution
hazardous_asteroids = df.loc[df["is_hazardous"], ["name", "velocity_km_s"]]


---

## 2.3. Calculations and simple statistics

In [ ]:
df.head()

In [ ]:
# 1. Estimate the approximate cross-sectional area (in km²)
#    This gives us an idea of how large each asteroid's "impact face" is.
df["area_km2"] = 3.14159 * (df["diameter_max_km"] / 2) ** 2

# 2. Compute a "momentum index" (velocity × diameter)
#    This is a simple, made-up indicator of potential impact force.
df["momentum_index"] = df["velocity_km_s"] * df["diameter_max_km"]

# 3. Compute a "danger score" — smaller distance, larger and faster asteroid = higher score
#    We divide by distance (km) and scale by 1e6 just for readability.
df["danger_score"] = (df["diameter_max_km"] * df["velocity_km_s"]) / (df["miss_distance_km"] + 1) * 1e6

In [ ]:
# Calculate common descriptive statistics for numeric columns of the whole dataframe
df.describe()

In [ ]:
# ... or for a single column
df["velocity_km_s"].describe()

In [ ]:
# or just a specific metric :-)

print("Velocity (km/s) statistics:")
print("Mean: ", df["velocity_km_s"].mean())
print("Median: ", df["velocity_km_s"].median())
print("Maximum: ", df["velocity_km_s"].max())
print("20% quantile: ", df["velocity_km_s"].quantile(0.2))
print("80% quantile: ", df["velocity_km_s"].quantile(0.8))

---
## <span style="color:#FF5D02;">Assignment: Compare descriptive statistics</span>

Compare the average velocity of small vs. large asteroids. Asteroids smaller than 0.05 would be considered small, above big. 
Which group travels faster on average? 

**Hints**

Use the `.mean()` function to calculate the mean

In [ ]:
# Solution
small = df[df["diameter_max_km"] < 0.05]
large = df[df["diameter_max_km"] >= 0.05]

mean_small = small["velocity_km_s"].mean()
mean_large = large["velocity_km_s"].mean()